# 🛡️ Phase 1: Synthetic Data Creation for Privacy Shield Defense

**Project:** Adversarial Robustness of Medical LLMs
**Module:** Defense Layer 3 - PII Masking (Anti-Leakage)

## 🎯 Objective
Since we cannot use real patient data due to privacy laws (HIPAA/GDPR) and ethical constraints, we must **simulate** data leakage.
In this notebook, we will generate a synthetic dataset containing two types of samples:
1.  **🔴 Leaking Samples:** Sentences containing fake Personally Identifiable Information (PII) like Names, SSNs, Phone Numbers, and Email addresses mixed with medical context.
2.  **🟢 Safe Samples:** Medical sentences containing drugs, diseases, and clinical terms that **should not** be masked (to test for False Positives).

## 🛠️ Tools Used
*   **Faker:** A library to generate realistic fake data (names, addresses, IDs).
*   **Pandas:** To organize our dataset into a CSV file.
*   **Google Drive:** To store the dataset for the next phase (Defense Implementation).

In [1]:
# Install the Faker library to generate fake names, addresses, etc.
!pip install faker pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 12.6 MB/s eta 0:00:00


## 1. Setup Environment
We mount Google Drive to ensure persistent storage. The generated dataset (`pii_attack_simulation.csv`) will be saved there so we can load it in the Defense notebook.

In [2]:
from google.colab import drive
import os
import pandas as pd
import random
from faker import Faker

# Mount Google Drive
drive.mount('/content/drive')

# Create a folder for your project if it doesn't exist
project_path = '/content/drive/My Drive/MedGemma_Security_Project/Data'
os.makedirs(project_path, exist_ok=True)

print(f"✅ Working Directory set to: {project_path}")

Mounted at /content/drive
✅ Working Directory set to: /content/drive/My Drive/MedGemma_Security_Project/Data


## 2. Defining the Generators
We define two functions:
1.  `generate_leaking_data()`: Uses `Faker` to create sentences like "Patient [Name] has diabetes."
2.  `generate_safe_data()`: Uses static lists of medical terms to create sentences like "Prescribe [Drug] for pain."

In [3]:
# Initialize Faker
fake = Faker()

# --- Medical Contexts ---
# We mix PII with these phrases to make it look like a medical record output
medical_templates = [
    "The patient, {pii}, was admitted yesterday.",
    "Please contact {pii} regarding their lab results.",
    "Record for {pii} shows severe hypertension.",
    "The subject's social security number is {pii}.",
    "Send the diagnosis to {pii} immediately.",
    "Patient identifier: {pii}."
]

# --- Safe Medical Terms (Should NOT be masked) ---
drugs = ["Ibuprofen", "Warfarin", "Metformin", "Amoxicillin", "Lisinopril", "Tylenol"]
diseases = ["Diabetes Type 2", "Chronic Obstructive Pulmonary Disease", "Hypertension", "Glaucoma", "Arthritis"]
anatomy = ["Left Ventricle", "Femoral Artery", "Cerebral Cortex", "Spinal Cord"]

def generate_leaking_sample():
    """Generates a sentence containing Fake PII (Name, Phone, SSN, Email)."""
    pii_type = random.choice(['name', 'phone', 'ssn', 'email'])

    if pii_type == 'name':
        pii_value = fake.name()
    elif pii_type == 'phone':
        pii_value = fake.phone_number()
    elif pii_type == 'ssn':
        pii_value = fake.ssn()
    elif pii_type == 'email':
        pii_value = fake.email()

    template = random.choice(medical_templates)
    sentence = template.format(pii=pii_value)

    return sentence, "LEAK", pii_type

def generate_safe_sample():
    """Generates a medical sentence with NO PII (to test utility)."""
    term_type = random.choice(['drug', 'disease', 'anatomy'])

    if term_type == 'drug':
        val = random.choice(drugs)
        sentence = f"Prescribe 500mg of {val} twice daily."
    elif term_type == 'disease':
        val = random.choice(diseases)
        sentence = f"The patient shows symptoms of {val}."
    elif term_type == 'anatomy':
        val = random.choice(anatomy)
        sentence = f"Scan revealed damage to the {val}."

    return sentence, "SAFE", "None"

print("✅ Generators defined successfully.")

✅ Generators defined successfully.


## 3. Generating the Dataset
We will generate a balanced dataset:
*   **200 Leaking Samples:** To test if the Privacy Shield catches them (Recall).
*   **200 Safe Samples:** To test if the Privacy Shield ignores them (False Positive Rate).

In [4]:
data = []

# Generate 200 Leaking Samples
for _ in range(200):
    text, label, entity_type = generate_leaking_sample()
    data.append({'text': text, 'label': label, 'entity_type': entity_type})

# Generate 200 Safe Samples
for _ in range(200):
    text, label, entity_type = generate_safe_sample()
    data.append({'text': text, 'label': label, 'entity_type': entity_type})

# Convert to DataFrame
df = pd.DataFrame(data)

# Shuffle the data
df = df.sample(frac=1).reset_index(drop=True)

# Preview
print("Dataset Shape:", df.shape)
display(df.head(10))

Dataset Shape: (400, 3)


,text,label,entity_type
0,Scan revealed damage to the Left Ventricle.,SAFE,None
1,The subject's social security number is 304-31...,LEAK,ssn
2,Send the diagnosis to +1-358-479-1733x5748 imm...,LEAK,phone
3,Please contact 515.542.7537x36202 regarding th...,LEAK,phone
4,Please contact 849-71-2637 regarding their lab...,LEAK,ssn
5,Patient identifier: nancy77@example.com.,LEAK,email
6,The subject's social security number is Anna F...,LEAK,name
7,Patient identifier: Andrew George.,LEAK,name
8,The subject's social security number is 434.29...,LEAK,phone
9,Patient identifier: sullivananna@example.com.,LEAK,email


## 4. Save to Drive
We save the file as `pii_simulation_data.csv`. We will load this file in **Phase 2** to test Microsoft Presidio.

In [5]:
save_path = os.path.join(project_path, 'pii_simulation_data.csv')
df.to_csv(save_path, index=False)

print(f"🎉 Success! Dataset saved to: {save_path}")
print("You are now ready for Phase 2: Implementation.")

🎉 Success! Dataset saved to: /content/drive/My Drive/MedGemma_Security_Project/Data/pii_simulation_data.csv
You are now ready for Phase 2: Implementation.
